# Multimodal RAG Pipeline — Amazon Product Dataset 2020

Builds a multimodal RAG system that indexes Amazon product **images** and **text** using **OpenAI CLIP** as the encoder, stores joint embeddings in **ChromaDB**, and answers queries with an open-source LLM served via the **Groq API**.

**Dataset**: [Amazon Product Dataset 2020](https://www.kaggle.com/datasets/promptcloud/amazon-product-dataset-2020)

| Part | Topic |
|------|-------|
| 0 | Setup & configuration |
| 1 | Load and preview the Amazon Product Dataset |
| 2 | CLIP encoder — image + text embeddings |
| 3 | Build the multimodal ChromaDB vector store |
| 4 | Multimodal retriever |
| 5 | LLM setup (Groq API) |
| 6 | Core multimodal RAG chain |
| 7 | Advanced query translation (Multi-Query, RAG-Fusion/RRF, Decomposition, Step-Back, HyDE) |
| 8 | Evaluation & strategy leaderboard |

---

## Part 0 — Setup & Configuration

In [1]:
# 0.1 — Install dependencies (run once, then restart the kernel)
%pip install -q \
    langchain langchain-community langchain-core \
    chromadb tiktoken \
    transformers accelerate bitsandbytes sentencepiece \
    torch torchvision Pillow requests \
    open-clip-torch \
    huggingface_hub \
    pandas kaggle tqdm \
    python-dotenv groq

# open-clip-torch ships CLIP weights without needing the full openai/clip repo


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 0.2 — API keys and directory setup
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current directory if it exists

DATA_DIR    = "./amazon_data"
IMAGE_DIR   = "./amazon_images"
PERSIST_DIR = "./chroma_amazon_mm"
COLLECTION  = "amazon_multimodal"

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(IMAGE_DIR,   exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)

# Prompt for Groq key if not already in environment
if not os.environ.get("GROQ_API_KEY"):
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

# Quick check — which keys are present
for key in ["GROQ_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"]:
    status = "✓" if os.environ.get(key) else "✗ MISSING"
    print(f"  {key}: {status}")

print("\nConfiguration loaded.")


  GROQ_API_KEY: ✓
  KAGGLE_USERNAME: ✓
  KAGGLE_KEY: ✓
  HF_TOKEN: ✓

Configuration loaded.


In [3]:
# 0.3 — Common imports
import json, os, warnings, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print("Imports OK")


Imports OK


---
## Part 1 — Load the Amazon Product Dataset

Download the dataset from Kaggle, keep only the columns we need, and optionally
download a sample of product images for the visual modality.

In [4]:
# 1.1 — Download dataset via the Kaggle CLI
import subprocess, zipfile

DATASET_SLUG = "promptcloud/amazon-product-dataset-2020"


def find_csv(root: Path) -> Path | None:
    # Walk the directory tree and return the first CSV found
    for p in sorted(root.rglob("*.csv")):
        return p
    return None


def extract_any_zips(directory: Path):
    # Kaggle sometimes skips auto-unzip, so we handle it manually
    for zf in directory.glob("*.zip"):
        print(f"  Extracting {zf.name}...")
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(directory)
        zf.unlink()


csv_path = find_csv(Path(DATA_DIR))

if csv_path is None:
    print("No CSV found locally — attempting Kaggle download...")

    # Try with --unzip first
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", DATA_DIR, "--unzip"],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip())
        print("Kaggle download finished (--unzip).")
    except Exception as e:
        print(f"  kaggle CLI error: {e}")
        # Fall back to manual extraction
        try:
            result = subprocess.run(
                ["kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", DATA_DIR],
                capture_output=True, text=True, timeout=300
            )
            print("Download complete. Extracting manually...")
            extract_any_zips(Path(DATA_DIR))
        except Exception as e2:
            print(f"  Second attempt failed: {e2}")

    csv_path = find_csv(Path(DATA_DIR))

    # Handle rare nested zip-in-zip case
    if csv_path is None:
        extract_any_zips(Path(DATA_DIR))
        csv_path = find_csv(Path(DATA_DIR))

else:
    print(f"Dataset already present: {csv_path}")


if csv_path is None:
    all_files = list(Path(DATA_DIR).rglob("*"))
    print("\nContents of DATA_DIR after download attempts:")
    for f in all_files:
        print(f"  {f}")
    print(
        "\n[ACTION REQUIRED] Automatic download failed.\n"
        "Download the dataset manually from:\n"
        "  https://www.kaggle.com/datasets/promptcloud/amazon-product-dataset-2020\n"
        f"and place the extracted CSV inside: {Path(DATA_DIR).resolve()}\n"
        "Then re-run this cell."
    )
    raise FileNotFoundError("CSV not found. See instructions above.")

print(f"\nUsing CSV: {csv_path}")
print(f"File size: {csv_path.stat().st_size / 1_000_000:.1f} MB")


No CSV found locally — attempting Kaggle download...
Kaggle download finished (--unzip).

Using CSV: amazon_data\home\sdf\marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv
File size: 19.6 MB


In [5]:
# 1.2 — Load and inspect the raw CSV
df_raw = pd.read_csv(csv_path, low_memory=False)
print(f"Rows: {len(df_raw):,}   Columns: {df_raw.shape[1]}")
print(df_raw.columns.tolist())


Rows: 10,002   Columns: 28
['Uniq Id', 'Product Name', 'Brand Name', 'Asin', 'Category', 'Upc Ean Code', 'List Price', 'Selling Price', 'Quantity', 'Model Number', 'About Product', 'Product Specification', 'Technical Details', 'Shipping Weight', 'Product Dimensions', 'Image', 'Variants', 'Sku', 'Product Url', 'Stock', 'Product Details', 'Dimensions', 'Color', 'Ingredients', 'Direction To Use', 'Is Amazon Seller', 'Size Quantity Variant', 'Product Description']


In [6]:
# 1.3 — Select and clean columns
# The dataset ships with slightly different column names across versions,
# so we map each canonical name to a list of possible raw names.
COLUMN_MAP = {
    "product_id":   ["Uniq Id", "uniq_id", "Product Id"],
    "product_name": ["Product Name", "product_name", "Title"],
    "category":     ["Category", "category"],
    "description":  ["About Product", "description", "Product Description"],
    "price":        ["Selling Price", "price"],
    "rating":       ["Product Rating", "rating"],
    "image_url":    ["Image", "image", "Image Url"],
    "brand":        ["Brand Name", "brand"],
}

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

rename = {}
for canonical, candidates in COLUMN_MAP.items():
    raw = find_col(df_raw, candidates)
    if raw:
        rename[raw] = canonical

df = df_raw.rename(columns=rename)[list(rename.values())].copy()

# Need both product name and image URL for multimodal indexing
df = df.dropna(subset=["product_name", "image_url"]).reset_index(drop=True)

for col in ["description", "category", "brand", "price", "rating"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

print(f"Clean rows: {len(df):,}")
df.head(3)


Clean rows: 10,002


,product_id,product_name,category,description,price,image_url,brand
0,4c69b61db1fc16e7013b43fc926e502d,"DB Longboards CoreFlex Crossbow 41"" Bamboo Fib...",Sports & Outdoors | Outdoor Recreation | Skate...,Make sure this fits by entering your model num...,$237.68,https://images-na.ssl-images-amazon.com/images...,
1,66d49bbed043f5be260fa9f7fbff5957,"Electronic Snap Circuits Mini Kits Classpack, ...",Toys & Games | Learning & Education | Science ...,Make sure this fits by entering your model num...,$99.95,https://images-na.ssl-images-amazon.com/images...,
2,2c55cae269aebf53838484b0d7dd931a,3Doodler Create Flexy 3D Printing Filament Ref...,Toys & Games | Arts & Crafts | Craft Kits,Make sure this fits by entering your model num...,$34.99,https://images-na.ssl-images-amazon.com/images...,


In [7]:
# 1.4 — Build product text strings for CLIP
# CLIP's text encoder caps at 77 tokens (~300-350 characters of typical product text).
# We pack fields in priority order and trim the description by word count to stay
# well within that limit — no tokenizer needed at this stage.

# Rough empirical limit: ~280 chars for the header fields leaves ~60 chars for description.
CLIP_HEADER_CHAR_LIMIT = 280
CLIP_DESC_WORD_LIMIT   = 15   # conservative; avoids any tokenizer dependency here


def make_product_text(row) -> str:
    parts = [f"Product: {str(row.get('product_name', ''))[:80].strip()}"]
    if row.get("brand"):    parts.append(f"Brand: {str(row['brand'])[:40]}")
    if row.get("category"): parts.append(f"Category: {str(row['category'])[:60]}")
    if row.get("price"):    parts.append(f"Price: {row['price']}")
    if row.get("rating"):   parts.append(f"Rating: {row['rating']}")

    header_text = " | ".join(parts)
    remaining_chars = CLIP_HEADER_CHAR_LIMIT - len(header_text)

    desc = str(row.get("description", "")).strip()
    if desc and remaining_chars > 20:
        # Estimate words from remaining characters (avg ~5 chars/word)
        max_words = max(1, min(CLIP_DESC_WORD_LIMIT, remaining_chars // 5))
        parts.append(f"Description: {' '.join(desc.split()[:max_words])}")

    return " | ".join(parts)


df["product_text"] = df.apply(make_product_text, axis=1)
print(f"product_text column created for {len(df):,} rows.")
print("Example:", df["product_text"].iloc[0])


product_text column created for 10,002 rows.
Example: Product: DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Complete | Category: Sports & Outdoors | Outdoor Recreation | Skates, Skateboards | Price: $237.68 | Description: Make sure this fits by entering your model number. | RESPONSIVE FLEX: The Crossbow features


In [8]:
# 1.5 — Download a sample of product images
# Images are optional but improve retrieval quality for visual queries.
# Failed downloads fall back to text-only embedding for that product.
import requests
from PIL import Image
from io import BytesIO

MAX_IMAGES = 10000   # increase to index more products
TIMEOUT_S  = 10
SAMPLE_DF  = df.head(MAX_IMAGES).copy()

image_paths = {}


def download_image(product_id, url, save_dir):
    dest = Path(save_dir) / f"{product_id}.jpg"
    if dest.exists():
        return str(dest)
    try:
        r = requests.get(url, timeout=TIMEOUT_S, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200 and r.headers.get("Content-Type", "").startswith("image"):
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.save(dest, "JPEG")
            return str(dest)
    except Exception:
        pass
    return None


print(f"Downloading up to {MAX_IMAGES} product images...")
for _, row in tqdm(SAMPLE_DF.iterrows(), total=len(SAMPLE_DF)):
    pid  = str(row.get("product_id", row.name))
    path = download_image(pid, row["image_url"], IMAGE_DIR)
    if path:
        image_paths[pid] = path

SAMPLE_DF["local_image"] = SAMPLE_DF.apply(
    lambda r: image_paths.get(str(r.get("product_id", r.name))), axis=1
)

n_images = SAMPLE_DF["local_image"].notna().sum()
print(f"\nImages downloaded: {n_images}/{len(SAMPLE_DF)}  ({n_images/len(SAMPLE_DF)*100:.1f}% success rate)")


  0%|          | 0/10000 [00:00<?, ?it/s]


Images downloaded: 9959/10000  (99.6% success rate)


---
## Part 2 — CLIP Multimodal Encoder

We use **OpenCLIP** (`open-clip-torch`) to encode both images and text into the same
512-dimensional embedding space, enabling true cross-modal search with a single index.

In [9]:
# 2.1 — Load CLIP model
import torch
import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"   # original OpenAI weights
CLIP_EMBED_DIM  = 512        # ViT-B/32 output dimension

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained=CLIP_PRETRAINED,
    device=DEVICE
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model.eval()

print(f"CLIP model loaded: {CLIP_MODEL_NAME} ({CLIP_PRETRAINED})  —  {CLIP_EMBED_DIM}-d embeddings")


Using device: cpu
CLIP model loaded: ViT-B-32 (openai)  —  512-d embeddings


In [10]:
# 2.2 — Encoder utilities

@torch.no_grad()
def encode_texts(texts: list[str], batch_size: int = 64) -> np.ndarray:
    # Encode text strings with CLIP's text encoder; returns L2-normalised embeddings
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i + batch_size]
        tokens = clip_tokenizer(batch).to(DEVICE)
        embs   = clip_model.encode_text(tokens)
        embs   = embs / embs.norm(dim=-1, keepdim=True)
        all_embeddings.append(embs.cpu().numpy())
    return np.vstack(all_embeddings).astype(np.float32)


@torch.no_grad()
def encode_images(image_paths_list: list[str | None], batch_size: int = 32) -> dict[int, np.ndarray]:
    # Encode local image files; returns {index -> embedding} for successful loads only
    results = {}
    valid   = [(i, p) for i, p in enumerate(image_paths_list) if p and Path(p).exists()]

    for start in range(0, len(valid), batch_size):
        batch = valid[start:start + batch_size]
        imgs, idxs = [], []
        for idx, path in batch:
            try:
                img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0)
                imgs.append(img)
                idxs.append(idx)
            except Exception:
                pass
        if imgs:
            stacked = torch.cat(imgs).to(DEVICE)
            embs    = clip_model.encode_image(stacked)
            embs    = embs / embs.norm(dim=-1, keepdim=True)
            for i, emb in zip(idxs, embs.cpu().numpy()):
                results[i] = emb.astype(np.float32)
    return results


def fuse_embeddings(text_emb: np.ndarray, image_emb: np.ndarray | None,
                    alpha: float = 0.5) -> np.ndarray:
    # Weighted average of text and image embeddings; falls back to text-only if no image
    if image_emb is None:
        return text_emb
    fused = (1 - alpha) * text_emb + alpha * image_emb
    norm  = np.linalg.norm(fused)
    return (fused / norm).astype(np.float32) if norm > 0 else fused


# Quick smoke-test
_t = encode_texts(["test query"])
print(f"Text encoder output shape: {_t.shape}  (should be (1, {CLIP_EMBED_DIM}))")
print("Encoder utilities ready.")


Text encoder output shape: (1, 512)  (should be (1, 512))
Encoder utilities ready.


---
## Part 3 — Build the Multimodal ChromaDB Vector Store

We wrap the CLIP encoder in a ChromaDB-compatible embedding function so LangChain's
standard `Chroma` integration works at query time.

In [11]:
# 3.1 — Custom CLIP embedding function for ChromaDB
import chromadb
from chromadb import EmbeddingFunction, Embeddings


class CLIPTextEmbeddingFunction(EmbeddingFunction):
    # Maps text queries into the same CLIP embedding space as the stored product vectors
    def __call__(self, input: list[str]) -> Embeddings:
        return encode_texts(input).tolist()


clip_embed_fn = CLIPTextEmbeddingFunction()
print("CLIPTextEmbeddingFunction ready.")


CLIPTextEmbeddingFunction ready.


In [12]:
# 3.2 — Index products into ChromaDB
# Each document stores the fused (text + image) CLIP embedding.
# Metadata carries the full product text so the LLM has context at retrieval time.

client = chromadb.PersistentClient(path=PERSIST_DIR)

# Drop and recreate the collection for a clean rebuild
try:
    client.delete_collection(COLLECTION)
    print(f"Deleted existing collection '{COLLECTION}'.")
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION,
    embedding_function=clip_embed_fn,
    metadata={"hnsw:space": "cosine"},
)

print(f"Encoding text for {len(SAMPLE_DF)} products...")
text_embeddings = encode_texts(SAMPLE_DF["product_text"].tolist())
print(f"Text embeddings shape: {text_embeddings.shape}")

print("Encoding images...")
image_emb_dict = encode_images(SAMPLE_DF["local_image"].tolist())
print(f"Image embeddings available for {len(image_emb_dict)}/{len(SAMPLE_DF)} products")

BATCH_SIZE   = 100
FUSION_ALPHA = 0.5  # equal weight for text and image

print("\nInserting into ChromaDB...")
for start in tqdm(range(0, len(SAMPLE_DF), BATCH_SIZE)):
    batch_df = SAMPLE_DF.iloc[start:start + BATCH_SIZE]
    ids, docs, metas, embs = [], [], [], []

    for local_idx, (df_idx, row) in enumerate(batch_df.iterrows()):
        global_idx = start + local_idx
        pid        = str(row.get("product_id", df_idx))
        text_emb   = text_embeddings[global_idx]
        image_emb  = image_emb_dict.get(global_idx)
        # Text-only products stay at alpha=0 so they stay in the pure text subspace
        alpha      = FUSION_ALPHA if image_emb is not None else 0.0
        fused      = fuse_embeddings(text_emb, image_emb, alpha=alpha)

        ids.append(pid)
        docs.append(row["product_text"])
        metas.append({
            "product_name": str(row.get("product_name", ""))[:200],
            "category":     str(row.get("category",     ""))[:100],
            "price":        str(row.get("price",        "")),
            "rating":       str(row.get("rating",       "")),
            "image_url":    str(row.get("image_url",    "")),
            "has_image":    str(image_emb is not None),
        })
        embs.append(fused.tolist())

    collection.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embs)

print(f"\nIndexed {collection.count():,} products into ChromaDB at '{PERSIST_DIR}'.")


Encoding text for 10000 products...
Text embeddings shape: (10000, 512)
Encoding images...
Image embeddings available for 9959/10000 products

Inserting into ChromaDB...


  0%|          | 0/100 [00:00<?, ?it/s]


Indexed 10,000 products into ChromaDB at './chroma_amazon_mm'.


---
## Part 4 — Multimodal Retriever

A thin wrapper around the ChromaDB collection that supports text queries, image
queries, and fused text+image queries.

In [13]:
# 4.1 — MultimodalRetriever
from langchain_core.documents import Document


class MultimodalRetriever:
    """
    Retrieves products from ChromaDB using CLIP embeddings.

    query_type options:
      'text'  — query is a string
      'image' — query is a PIL Image or a local file path
      'fused' — query is (text_string, image_or_path); embeddings are averaged
    """

    def __init__(self, chroma_collection, k: int = 6, fusion_alpha: float = 0.5):
        self.collection   = chroma_collection
        self.k            = k
        self.fusion_alpha = fusion_alpha

    def _text_emb(self, text: str) -> list[float]:
        return encode_texts([text])[0].tolist()

    def _image_emb(self, image) -> list[float]:
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.encode_image(tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.cpu().numpy()[0].astype(np.float32).tolist()

    def _fuse(self, t_emb, i_emb):
        t = np.array(t_emb, dtype=np.float32)
        i = np.array(i_emb, dtype=np.float32)
        fused = (1 - self.fusion_alpha) * t + self.fusion_alpha * i
        norm  = np.linalg.norm(fused)
        return (fused / norm).tolist() if norm > 0 else fused.tolist()

    def retrieve(self, query, query_type: str = "text") -> list[Document]:
        if query_type == "text":
            emb = self._text_emb(query)
        elif query_type == "image":
            emb = self._image_emb(query)
        elif query_type == "fused":
            text, image = query
            emb = self._fuse(self._text_emb(text), self._image_emb(image))
        else:
            raise ValueError(f"Unknown query_type: {query_type!r}")

        results = self.collection.query(
            query_embeddings=[emb],
            n_results=self.k,
            include=["documents", "metadatas", "distances"]
        )

        docs = []
        for doc_text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            docs.append(Document(
                page_content=doc_text,
                metadata={**meta, "cosine_distance": round(dist, 4)}
            ))
        return docs

    def rerank_by_image(self, query_image_pil, docs: list[Document],
                        image_dir: str, top_n: int | None = None) -> list[Document]:
        # Second-pass rerank using direct image-to-image cosine similarity.
        # Products without a local image keep their original retrieval score.
        if top_n is None:
            top_n = self.k
        if isinstance(query_image_pil, str):
            query_image_pil = Image.open(query_image_pil).convert("RGB")

        tensor = clip_preprocess(query_image_pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            q_emb = clip_model.encode_image(tensor)
            q_emb = (q_emb / q_emb.norm(dim=-1, keepdim=True)).cpu().numpy()[0].astype(np.float32)

        scored = []
        for doc in docs:
            img_url  = doc.metadata.get("image_url", "")
            img_name = Path(img_url).stem if img_url else ""
            img_path = Path(image_dir) / f"{img_name}.jpg"

            if img_path.exists():
                try:
                    img_tensor = clip_preprocess(
                        Image.open(img_path).convert("RGB")
                    ).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        i_emb = clip_model.encode_image(img_tensor)
                        i_emb = (i_emb / i_emb.norm(dim=-1, keepdim=True)).cpu().numpy()[0].astype(np.float32)
                    sim = float(np.dot(q_emb, i_emb))
                    scored.append((sim, doc))
                    continue
                except Exception:
                    pass
            # No local image — use the stored cosine similarity as the fallback score
            fallback = 1.0 - doc.metadata.get("cosine_distance", 0.5)
            scored.append((fallback, doc))

        scored.sort(key=lambda x: -x[0])
        return [d for _, d in scored[:top_n]]

    def invoke(self, query: str) -> list[Document]:
        # LangChain-compatible interface for plain text queries
        return self.retrieve(query, query_type="text")


retriever = MultimodalRetriever(collection, k=6)

test_docs = retriever.invoke("wireless noise-cancelling headphones")
print(f"Retrieved {len(test_docs)} products for test query.")
for d in test_docs[:3]:
    print(f"  [{d.metadata['cosine_distance']:.3f}] {d.metadata['product_name'][:70]}")


Retrieved 6 products for test query.
  [0.349] Board Games and Beer T Shirt For Gamer and Drinker
  [0.357] I'd Rather Be Playing Board Games
  [0.365] I Read Rules for Board Games and I Like It T-Shirt


---
## Part 5 — LLM Setup (Groq API)

We use the **Groq API** with `llama-3.1-8b-instant` for fast, free-tier inference.
Swap `LLM_MODEL_ID` for any other Groq-supported model if you prefer.

In [14]:
# 5.1 — LLM configuration
LLM_MODEL_ID   = "llama-3.1-8b-instant"
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.1

print(f"Model      : {LLM_MODEL_ID}")
print(f"Max tokens : {MAX_NEW_TOKENS}")
print(f"Temperature: {TEMPERATURE}")


Model      : llama-3.1-8b-instant
Max tokens : 512
Temperature: 0.1


In [15]:
# 5.2 — Groq client and chat helper
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


def hf_chat_complete(messages: list[dict],
                     max_new_tokens: int = MAX_NEW_TOKENS,
                     temperature: float = TEMPERATURE) -> str:
    # Thin wrapper around the Groq chat completions endpoint
    resp = groq_client.chat.completions.create(
        model=LLM_MODEL_ID,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


# Quick smoke-test
test_reply = hf_chat_complete([{"role": "user", "content": "Reply with exactly: OK"}])
print(f"LLM smoke-test: '{test_reply}'")


LLM smoke-test: 'OK'


In [16]:
# 5.3 — Optional: local transformers pipeline (needs ≥16 GB VRAM)
# Skip this cell unless you want to run the model locally instead of via Groq.

LOCAL_LLM_ENABLED = False   # set to True to enable

if LOCAL_LLM_ENABLED:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

    LOCAL_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer  = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID, token=os.environ.get("HF_TOKEN"))
    local_llm  = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        token=os.environ.get("HF_TOKEN"),
        quantization_config=bnb_config,
        device_map="auto",
    )
    local_pipe = pipeline(
        "text-generation",
        model=local_llm,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
    )

    def hf_chat_complete(messages, **kwargs):
        # Override the Groq function with a local pipeline call
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        out = local_pipe(prompt)
        return out[0]["generated_text"][len(prompt):].strip()

    print(f"Local {LOCAL_MODEL_ID} loaded (4-bit quantised).")
else:
    print("Local pipeline disabled — using Groq API.")


Local pipeline disabled — using Groq API.


---
## Part 6 — Core Multimodal RAG Chain

Retrieve top-k products → format as context → generate an answer with the LLM.

In [17]:
# 6.1 — System prompt and context formatter

RAG_SYSTEM_PROMPT = """You are a helpful Amazon product assistant.
Use ONLY the product information below to answer the user's question.
If the answer is not in the provided products, say exactly:
  "I don't have that information in the product catalogue."
Do not make up product names, prices, or features.
When recommending products, briefly explain why each matches the query."""


def format_product_docs(docs: list[Document], max_chars_per_product: int = 300) -> str:
    # Truncates each product to avoid overflowing the LLM context window
    parts = []
    for i, doc in enumerate(docs, 1):
        meta    = doc.metadata
        img     = meta.get("image_url", "N/A")
        content = doc.page_content[:max_chars_per_product]
        part    = f"[Product {i}]\n{content}"
        if img and img != "N/A":
            part += f"\nImage: {img}"
        parts.append(part)
    return "\n\n---\n\n".join(parts)


def rag_answer(query: str, query_type: str = "text",
               image_query=None, k: int = 12) -> str:
    retr = MultimodalRetriever(collection, k=k)

    if query_type == "text":
        docs = retr.retrieve(query, query_type="text")
    elif query_type == "image":
        docs = retr.retrieve(image_query, query_type="image")
    elif query_type == "fused":
        docs = retr.retrieve((query, image_query), query_type="fused")
    else:
        raise ValueError(f"Unknown query_type: {query_type!r}")

    context  = format_product_docs(docs)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    return hf_chat_complete(messages)


print("RAG chain ready.")


RAG chain ready.


In [18]:
# 6.2 — Text query demos

q1 = "What are some good wireless headphones under $50?"
print(f"Q: {q1}\n")
print(rag_answer(q1))


Q: What are some good wireless headphones under $50?

Based on the provided product catalogue, I recommend the following wireless headphones under $50:

1. Product 2: Trolls Poppy Kid Friendly Headphones - These headphones are kid-friendly, have a built-in volume limiting feature, and are priced at $17.99, which is under $50.

I don't have any other wireless headphones under $50 in the product catalogue.


In [19]:
q2 = "Recommend a laptop bag suitable for a 15-inch laptop with good reviews."
print(f"Q: {q2}\n")
print(rag_answer(q2))


Q: Recommend a laptop bag suitable for a 15-inch laptop with good reviews.

I don't have that information in the product catalogue.


In [20]:
# 6.3 — Image-query demo
# Picks any locally saved product image and retrieves visually similar items

sample_image_path = next(iter(image_paths.values()), None)

if sample_image_path:
    print(f"Image query using: {sample_image_path}")
    docs = retriever.retrieve(sample_image_path, query_type="image")
    print(f"Retrieved {len(docs)} visually similar products:")
    for d in docs[:4]:
        print(f"  [{d.metadata['cosine_distance']:.3f}] {d.metadata['product_name'][:70]}")
else:
    print("No images downloaded — skipping image query demo.")


Image query using: amazon_images\4c69b61db1fc16e7013b43fc926e502d.jpg
Retrieved 6 visually similar products:
  [0.192] DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Comple
  [0.274] Rayne Longboards Demonseed Longboard Complete
  [0.277] DB Longboards Phase 38" Maple Drop Through Longboard
  [0.280] SWAGSKATE NG2 A.I.-Powered Electric Longboard with Hands-Free Control 


---
## Part 7 — Advanced Query Translation

Five retrieval strategies that go beyond a single embedding lookup.

In [21]:
# 7.0 — Shared helpers

def llm_complete(user_prompt: str, system_prompt: str = "") -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return hf_chat_complete(messages)


def get_unique_union(docs_lists: list[list[Document]]) -> list[Document]:
    # Merge multiple result lists, dropping duplicates by content prefix
    seen, unique = set(), []
    for docs in docs_lists:
        for doc in docs:
            key = doc.page_content[:120]
            if key not in seen:
                seen.add(key)
                unique.append(doc)
    return unique


def reciprocal_rank_fusion(results_lists: list[list[Document]],
                           k: int = 60) -> list[tuple[Document, float]]:
    # RRF score = sum(1 / (k + rank)) across all result lists
    scores, lookup = {}, {}
    for results in results_lists:
        for rank, doc in enumerate(results):
            key = doc.page_content[:120]
            lookup[key] = doc
            scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [(lookup[k], v) for k, v in ranked]


print("Shared utilities ready.")


Shared utilities ready.


In [22]:
# 7.1 — Multi-Query

def run_multi_query(query: str, n_alt: int = 4) -> tuple[list[Document], list[str]]:
    prompt = (
        f"Generate {n_alt} alternative phrasings of this Amazon product search query.\n"
        "Output one query per line, no numbering or bullet points.\n"
        f"\nOriginal query: {query}"
    )
    raw          = llm_complete(prompt)
    alternatives = [q.strip() for q in raw.strip().splitlines() if q.strip()][:n_alt]
    all_queries  = [query] + alternatives
    docs_lists   = [retriever.invoke(q) for q in all_queries]
    unique_docs  = get_unique_union(docs_lists)
    return unique_docs, all_queries


mq_query          = "budget smartphone with good camera"
mq_docs, mq_queries = run_multi_query(mq_query)
print(f"Original   : {mq_query}")
print(f"Alternatives: {mq_queries[1:]}")
print(f"Standard: {len(retriever.invoke(mq_query))} chunks  |  Multi-query: {len(mq_docs)} unique chunks")

context = format_product_docs(mq_docs)
mq_ans  = hf_chat_complete([
    {"role": "system", "content": RAG_SYSTEM_PROMPT},
    {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {mq_query}"},
])
print(f"\nMulti-Query Answer:\n{mq_ans}")


Original   : budget smartphone with good camera
Alternatives: ['Affordable smartphone with high-quality camera capabilities', 'Cheap smartphone options with excellent camera performance', 'Budget-friendly mobile phone with good camera quality', 'Low-cost smartphone with a great camera feature']
Standard: 6 chunks  |  Multi-query: 9 unique chunks

Multi-Query Answer:
I don't have that information in the product catalogue.


In [23]:
# 7.2 — RAG-Fusion with Reciprocal Rank Fusion

def run_rag_fusion(query: str, n_alt: int = 3, top_k: int = 12):
    prompt       = (
        f"Generate {n_alt} alternative Amazon product search queries for:\n"
        f"{query}\nOne per line."
    )
    alternatives = [q.strip() for q in llm_complete(prompt).strip().splitlines() if q.strip()][:n_alt]
    all_queries  = [query] + alternatives
    docs_lists   = [retriever.invoke(q) for q in all_queries]
    rrf_results  = reciprocal_rank_fusion(docs_lists)
    top_docs     = [doc for doc, _ in rrf_results[:top_k]]
    return top_docs, rrf_results


rrf_query          = "ergonomic office chair with lumbar support"
rrf_docs, rrf_ranked = run_rag_fusion(rrf_query)
print(f"Query: {rrf_query}")
print("Top products by RRF score:")
for doc, score in rrf_ranked[:5]:
    print(f"  [RRF={score:.4f}] {doc.metadata['product_name'][:60]}")

rrf_ans = hf_chat_complete([
    {"role": "system", "content": RAG_SYSTEM_PROMPT},
    {"role": "user",   "content": f"Context:\n{format_product_docs(rrf_docs)}\n\nQuestion: {rrf_query}"},
])
print(f"\nRAG-Fusion Answer:\n{rrf_ans}")


Query: ergonomic office chair with lumbar support
Top products by RRF score:
  [RRF=0.0656] Flash Furniture Black Padded Ergonomic Shell Chair with Left
  [RRF=0.0638] Portable Multiuse Adjustable Recliner Stadium Seat by Tradem
  [RRF=0.0635] Board Games and Beer T Shirt For Gamer and Drinker
  [RRF=0.0628] Shapes Structured Vinyl Soft Seating with Durable Frame- Rou
  [RRF=0.0159] Flash Furniture 2 Pk. Band/Music Stack Chair Dolly

RAG-Fusion Answer:
I don't have that information in the product catalogue.


In [24]:
# 7.3 — Query Decomposition

def run_decomposition(query: str) -> tuple[str, list[tuple[str, str]]]:
    decomp_prompt = (
        "Break this Amazon product question into 3 simpler sub-questions.\n"
        "One per line, no numbering.\n"
        f"Question: {query}"
    )
    sub_questions = [q.strip() for q in llm_complete(decomp_prompt).strip().splitlines() if q.strip()][:4]

    sub_answers = []
    for sq in sub_questions:
        docs = retriever.invoke(sq)
        ans  = hf_chat_complete([
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{format_product_docs(docs)}\n\nQuestion: {sq}"},
        ])
        sub_answers.append((sq, ans))

    synthesis_text = "\n".join(f"Q: {q}\nA: {a}" for q, a in sub_answers)
    final = hf_chat_complete([
        {"role": "system", "content": "Synthesize sub-answers into one coherent answer."},
        {"role": "user",   "content": f"Original question: {query}\n\n{synthesis_text}"},
    ])
    return final, sub_answers


d_query = "What are the best gifts for a tech-savvy person who works from home?"
print(f"Complex question: {d_query}")
d_ans, d_subs = run_decomposition(d_query)
print("\nSub-questions:")
for sq, sa in d_subs:
    print(f"  Q: {sq}")
    print(f"  A: {sa[:150]}...")
print(f"\nFinal answer:\n{d_ans}")


Complex question: What are the best gifts for a tech-savvy person who works from home?

Sub-questions:
  Q: What are some popular gift ideas for tech-savvy individuals?
  A: I don't have that information in the product catalogue....
  Q: What types of products are useful for remote workers or those who work from home?
  A: Based on the provided product catalogue, the following products could be useful for remote workers or those who work from home:

1. [Product 1] - Boar...
  Q: Are there any high-tech gadgets that can enhance productivity or comfort for someone who works from home?
  A: I don't have that information in the product catalogue....

Final answer:
Based on the provided information, it seems that the product catalogue does not have a wide range of gift ideas for tech-savvy individuals or high-tech gadgets that can enhance productivity or comfort for remote workers. However, there are a few suggestions that could be useful or fun for someone who works from home:

While not d

In [25]:
# 7.4 — Step-Back Prompting

def run_step_back(query: str) -> tuple[str, str, list[Document]]:
    abstract_prompt = (
        "Rewrite this specific product question as a broader product category question.\n"
        "Examples:\n"
        "  Specific: 'Sony WH-1000XM5 headphones'   → Abstract: 'noise-cancelling headphones'\n"
        "  Specific: 'best standing desk under $300' → Abstract: 'ergonomic office furniture'\n"
        f"\nSpecific: {query}\nAbstract:"
    )
    abstract_query = llm_complete(abstract_prompt).strip().split("\n")[0].strip()

    specific_docs = retriever.invoke(query)
    abstract_docs = retriever.invoke(abstract_query)
    combined_docs = get_unique_union([specific_docs, abstract_docs])

    answer = hf_chat_complete([
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{format_product_docs(combined_docs)}\n\nQuestion: {query}"},
    ])
    return answer, abstract_query, combined_docs


sb_query = "Is there a waterproof Bluetooth speaker that works well outdoors?"
sb_ans, sb_abstract, sb_docs = run_step_back(sb_query)
print(f"Original : {sb_query}")
print(f"Abstract : {sb_abstract}")
print(f"Docs used: {len(sb_docs)}")
print(f"\nStep-Back Answer:\n{sb_ans}")


Original : Is there a waterproof Bluetooth speaker that works well outdoors?
Abstract : Specific: Is there a waterproof Bluetooth speaker that works well outdoors?
Docs used: 7

Step-Back Answer:
I don't have that information in the product catalogue.


In [26]:
# 7.5 — HyDE (Hypothetical Document Embeddings)
# Ask the LLM to write a hypothetical product listing that would answer the query,
# then embed that listing with CLIP to find the real products closest to it.

def run_hyde(query: str) -> tuple[str, str, list[Document], list[Document]]:
    hyde_prompt = (
        "Write a short Amazon product listing (name, category, key features, price range) "
        "that would perfectly answer this query. Be specific and realistic.\n"
        f"\nQuery: {query}\n\nHypothetical product listing:"
    )
    hypothetical = llm_complete(hyde_prompt).strip()
    hypo_short   = " ".join(hypothetical.split()[:60])  # stay within CLIP's ~77 token limit

    hypo_emb = encode_texts([hypo_short])[0].tolist()
    raw_hits = collection.query(
        query_embeddings=[hypo_emb], n_results=12,
        include=["documents", "metadatas", "distances"]
    )
    hyde_docs = [
        Document(
            page_content=doc,
            metadata={**meta, "cosine_distance": round(dist, 4)}
        )
        for doc, meta, dist in zip(
            raw_hits["documents"][0],
            raw_hits["metadatas"][0],
            raw_hits["distances"][0],
        )
    ]

    baseline_docs = retriever.invoke(query)

    answer = hf_chat_complete([
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{format_product_docs(hyde_docs)}\n\nQuestion: {query}"},
    ])
    return answer, hypothetical, hyde_docs, baseline_docs


hyde_query = "I need a portable projector for movie nights in my backyard."
hyde_ans, hyde_hypo, hyde_docs, hyde_base = run_hyde(hyde_query)
print(f"Query: {hyde_query}")
print(f"\nHypothetical product snippet:\n{hyde_hypo[:300]}")
print(f"\nStandard: {len(hyde_base)} docs  |  HyDE: {len(hyde_docs)} docs")
print(f"\nHyDE Answer:\n{hyde_ans}")


Query: I need a portable projector for movie nights in my backyard.

Hypothetical product snippet:
**Product Name:** YOUPRO 1080p Outdoor Portable Projector

**Category:** Electronics > Home Audio & Theater > Projectors

**Key Features:**

- **Bright and Clear Image**: 3,500 lumens of brightness for a vibrant image in outdoor settings
- **Portability**: Weighs only 3.5 lbs and comes with a carryi

Standard: 6 docs  |  HyDE: 12 docs

HyDE Answer:
I don't have that information in the product catalogue.


---
## Part 8 — Evaluation & Strategy Leaderboard

Generate test questions from indexed products, run all six retrieval strategies,
and auto-grade answers with LLM-as-judge.

In [27]:
# 8.1 — Generate test questions from the indexed products
import re

N_TEST_QUESTIONS = 10

sample_products = SAMPLE_DF["product_name"].dropna().drop_duplicates().sample(
    min(N_TEST_QUESTIONS * 2, len(SAMPLE_DF)), random_state=42, replace=False
).tolist()

gen_prompt = (
    "You are creating evaluation questions for an Amazon product RAG system.\n"
    "Based on these real product names from the catalogue:\n"
    + "\n".join(f"- {p[:80]}" for p in sample_products[:15])
    + f"\n\nGenerate {N_TEST_QUESTIONS} diverse user questions that would lead to "
    "finding one of these products. Each question should be natural and feature-focused "
    "(do NOT just ask for the product name directly).\n"
    "The 'expected_product' field must be the EXACT product name from the list above.\n"
    "Include a mix of feature-focused, price-focused, use-case, and category questions.\n"
    "Output ONLY a valid JSON array -- no explanation, no markdown.\n"
    'Each element must have exactly two keys: \"question\" and \"expected_product\".\n'
    'Example: [{"question": "Affordable party cups for kids?", "expected_product": "Hot Wheels Wild Racer Cups, 9 oz., Party Favor"}]'
)

raw_qs = llm_complete(gen_prompt)


def extract_json_array(text):
    # Strip markdown fences and try straight JSON parse first
    text = re.sub(r"```(?:json)?", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Try to find the outermost array
    match = re.search(r"(\[.*?\])", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    # Last resort: extract individual objects
    objects = re.findall(r'\{[^{}]+\}', text)
    results = []
    for obj in objects:
        try:
            results.append(json.loads(obj))
        except json.JSONDecodeError:
            continue
    if results:
        return results
    raise ValueError(f"Could not extract JSON array from LLM output:\n{text[:500]}")


test_questions = extract_json_array(raw_qs)

print(f"Generated {len(test_questions)} test questions.")
for tq in test_questions[:3]:
    print(f"  Q: {tq['question']}")
    print(f"  Expected: {tq['expected_product']}")
    print()


Generated 10 test questions.
  Q: Looking for a model of a farm shop for my HO scale model train set?
  Expected: Busch 1512 Farm Shop HO Structure Scale Model Structure

  Q: What's a good lunch bag for school that's easy to clean?
  Expected: Everest Cooler/Lunch Bag, Orange/White Dot

  Q: Want a fun design for my hoverboard?
  Expected: MightySkins Skin Compatible with Self Balancing Mini Scooter Hover Board - Life



In [28]:
# 8.2 — Unified strategy runner

STRATEGIES = ["standard", "multi_query", "rag_fusion", "decomposition", "step_back", "hyde"]


def run_strategy(strategy_name: str, query: str) -> dict:
    result = {"answer": "", "chunks_used": 0, "extra": {}, "error": None}
    try:
        if strategy_name == "standard":
            docs   = retriever.invoke(query)
            answer = hf_chat_complete([
                {"role": "system", "content": RAG_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n{format_product_docs(docs)}\n\nQuestion: {query}"},
            ])
            result.update(answer=answer, chunks_used=len(docs))

        elif strategy_name == "multi_query":
            docs, all_q = run_multi_query(query, n_alt=4)
            answer = hf_chat_complete([
                {"role": "system", "content": RAG_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n{format_product_docs(docs)}\n\nQuestion: {query}"},
            ])
            result.update(answer=answer, chunks_used=len(docs),
                          extra={"queries_generated": all_q[1:]})

        elif strategy_name == "rag_fusion":
            docs, rrf = run_rag_fusion(query, n_alt=3, top_k=12)
            answer = hf_chat_complete([
                {"role": "system", "content": RAG_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Context:\n{format_product_docs(docs)}\n\nQuestion: {query}"},
            ])
            result.update(answer=answer, chunks_used=len(docs),
                          extra={"top_rrf_scores": [round(s, 4) for _, s in rrf[:3]]})

        elif strategy_name == "decomposition":
            answer, subs = run_decomposition(query)
            result.update(answer=answer, chunks_used=len(subs) * 6,
                          extra={"sub_questions": [q for q, _ in subs]})

        elif strategy_name == "step_back":
            answer, abstract_q, docs = run_step_back(query)
            result.update(answer=answer, chunks_used=len(docs),
                          extra={"abstract_query": abstract_q})

        elif strategy_name == "hyde":
            answer, hypo, docs, _ = run_hyde(query)
            result.update(answer=answer, chunks_used=len(docs),
                          extra={"hypothetical_snippet": hypo[:200]})

    except Exception as exc:
        result["error"] = traceback.format_exc(limit=3)
        print(f"    ⚠ {strategy_name} error: {exc}")
    return result


print("Strategy runner ready.")


Strategy runner ready.


In [29]:
# 8.3 — Run all strategies across every test question
all_results = []

print(f"Running {len(test_questions)} × {len(STRATEGIES)} = "
      f"{len(test_questions) * len(STRATEGIES)} total calls.\n")
print("=" * 70)

for q_idx, qa in enumerate(test_questions):
    question = qa["question"]
    expected = qa.get("expected_product", "")
    print(f"\n[Q{q_idx+1}/{len(test_questions)}] {question[:80]}")

    for strategy in STRATEGIES:
        print(f"  ▶ {strategy:<14}", end=" ", flush=True)
        t0      = time.time()
        run     = run_strategy(strategy, question)
        elapsed = round(time.time() - t0, 1)
        status  = "✓" if not run["error"] else "✗"
        print(f"{status}  ({elapsed}s, {run['chunks_used']} chunks)")

        all_results.append({
            "q_index":     q_idx + 1,
            "question":    question,
            "expected":    expected,
            "strategy":    strategy,
            "answer":      run["answer"],
            "chunks_used": run["chunks_used"],
            "elapsed_s":   elapsed,
            "extra":       run["extra"],
            "error":       run["error"],
            "correctness": None,
            "helpfulness": None,
            "hallucination": None,
        })

print(f"\nDone. {len(all_results)} runs collected.")


Running 10 × 6 = 60 total calls.


[Q1/10] Looking for a model of a farm shop for my HO scale model train set?
  ▶ standard       ✓  (16.5s, 6 chunks)
  ▶ multi_query    ✓  (23.1s, 12 chunks)
  ▶ rag_fusion     ✓  (21.3s, 12 chunks)
  ▶ decomposition  ✓  (37.5s, 18 chunks)
  ▶ step_back      ✓  (21.5s, 9 chunks)
  ▶ hyde               ⚠ hyde error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01krmqe39ye3arrjh598tsffgc` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4049, Requested 2140. Please try again in 1.89s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
✗  (18.0s, 0 chunks)

[Q2/10] What's a good lunch bag for school that's easy to clean?
  ▶ standard       ✓  (0.6s, 6 chunks)
  ▶ multi_query    ✓  (24.5s, 16 chunks)
  ▶ rag_fusion     ✓  (21.7s, 12 chunks)
  ▶ decomposition  ✓  (34.9s, 18 chunks

In [30]:
# 8.4 — LLM-as-judge grading
# Three independent 0-10 scores per answer:
#   correctness  — did we find the right product (or a fair equivalent)?
#   helpfulness  — is the answer actually useful to the user?
#   hallucination — are all stated facts grounded in the retrieved context?

GRADE_SYSTEM = (
    "You are an expert evaluator for a product recommendation RAG system. "
    "Score the answer on three INDEPENDENT criteria, each 0-10. "
    "Respond ONLY with a JSON object -- no markdown, no extra text."
)


def grade_answer(question, expected, rag_answer_text):
    prompt = (
        f"Question:          {question}\n"
        f"Expected product:  {expected}\n"
        f"RAG Answer:        {rag_answer_text}\n\n"
        "Score each criterion 0-10 INDEPENDENTLY:\n\n"
        "correctness (0-10):\n"
        "  10 = answer recommends the expected product or a clearly equivalent one\n"
        "   7 = answer recommends a very similar product in the same sub-category\n"
        "   4 = answer is in the right general category but not a close match\n"
        "   0 = answer is completely off-topic or says no product was found\n\n"
        "helpfulness (0-10):\n"
        "  10 = very helpful with clear reasons why products match the query\n"
        "   5 = somewhat helpful, mentions relevant products\n"
        "   0 = not helpful at all\n\n"
        "hallucination (0-10):\n"
        "  10 = no fabricated details — all prices/features/names match context\n"
        "   5 = minor embellishments or vague claims\n"
        "   0 = major fabrications (invented prices, features, or product names)\n"
        "  NOTE: recommending a different real product is NOT hallucination.\n\n"
        'Respond ONLY: {"correctness": <0-10>, "helpfulness": <0-10>, "hallucination": <0-10>, "reason": "<brief>"}'
    )
    try:
        raw   = hf_chat_complete([{"role": "system", "content": GRADE_SYSTEM},
                                   {"role": "user",   "content": prompt}])
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = json.loads(clean)
        for key in ("correctness", "helpfulness", "hallucination"):
            if isinstance(parsed.get(key), (int, float)):
                parsed[key] = max(0, min(10, parsed[key]))
        return parsed
    except json.JSONDecodeError:
        scores = {}
        for key in ("correctness", "helpfulness", "hallucination"):
            m = re.search(rf'"{key}"\s*:\s*(\d+(?:\.\d+)?)', raw)
            scores[key] = float(m.group(1)) if m else None
        scores["reason"] = raw[:200]
        return scores
    except Exception as e:
        return {"correctness": None, "helpfulness": None, "hallucination": None, "reason": str(e)}


print(f"Grading {len(all_results)} answers...\n")
for i, record in enumerate(all_results):
    if record["error"]:
        continue
    scores = grade_answer(record["question"], record["expected"], record["answer"])
    record["correctness"]   = scores.get("correctness")
    record["helpfulness"]   = scores.get("helpfulness")
    record["hallucination"] = scores.get("hallucination")
    record["grade_reason"]  = scores.get("reason", "")
    if (i + 1) % len(STRATEGIES) == 0:
        print(f"  Q{record['q_index']} graded.")

print("\nGrading complete.")


Grading 60 answers...

  Q2 graded.
  Q3 graded.
  Q4 graded.
  Q5 graded.
  Q6 graded.
  Q7 graded.
  Q8 graded.
  Q9 graded.
  Q10 graded.

Grading complete.


In [31]:
# 8.5 — Scorecard & Leaderboard
# overall = correctness*0.40 + helpfulness*0.35 + hallucination*0.25

def avg(vals):
    clean = [v for v in vals if v is not None]
    return round(sum(clean) / len(clean), 2) if clean else None

WEIGHTS = {"correctness": 0.40, "helpfulness": 0.35, "hallucination": 0.25}

scorecard = {}
for strategy in STRATEGIES:
    rows = [r for r in all_results if r["strategy"] == strategy and not r["error"]]
    sc = {
        "strategy":      strategy,
        "n_questions":   len(rows),
        "correctness":   avg([r["correctness"]   for r in rows]),
        "helpfulness":   avg([r["helpfulness"]   for r in rows]),
        "hallucination": avg([r["hallucination"] for r in rows]),
        "avg_chunks":    avg([r["chunks_used"]   for r in rows]),
        "avg_latency_s": avg([r["elapsed_s"]     for r in rows]),
    }
    sc["overall"] = round(sum(sc[k] * w for k, w in WEIGHTS.items() if sc[k] is not None), 2)
    scorecard[strategy] = sc

ranked = sorted(scorecard.values(), key=lambda x: x["overall"] or 0, reverse=True)

print("\n" + "=" * 75)
print("STRATEGY LEADERBOARD")
print("=" * 75)
print(f"{'Rank':<5} {'Strategy':<16} {'Overall':>7} {'Correct':>8} {'Helpful':>8} {'No-Halluc':>10} {'Chunks':>7} {'Latency':>8}")
print("-" * 75)
for rank, sc in enumerate(ranked, 1):
    print(
        f"#{rank:<4} {sc['strategy']:<16} "
        f"{str(sc['overall']):>7} "
        f"{str(sc['correctness']):>8} "
        f"{str(sc['helpfulness']):>8} "
        f"{str(sc['hallucination']):>10} "
        f"{str(sc['avg_chunks']):>7} "
        f"{str(sc['avg_latency_s']):>7}s"
    )
print("=" * 75)
print("Scores out of 10.  overall = correctness*0.40 + helpfulness*0.35 + hallucination*0.25")

out_dir = Path(DATA_DIR)
with open(out_dir / "multimodal_rag_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
with open(out_dir / "multimodal_rag_leaderboard.json", "w") as f:
    json.dump({"leaderboard": ranked}, f, indent=2)

print(f"\nResults saved to {out_dir}/")



STRATEGY LEADERBOARD
Rank  Strategy         Overall  Correct  Helpful  No-Halluc  Chunks  Latency
---------------------------------------------------------------------------
#1    rag_fusion          7.48      6.5      6.8       10.0    11.3   21.64s
#2    hyde                7.24     6.11     6.56       10.0    12.0   25.04s
#3    multi_query         6.83      5.5      6.1       10.0    12.0   22.76s
#4    standard            6.33     4.22     6.11       10.0     6.0   11.57s
#5    step_back           6.29      4.4      5.8       10.0     9.0   18.42s
#6    decomposition       5.11     2.44     4.67       10.0    18.0    38.4s
Scores out of 10.  overall = correctness*0.40 + helpfulness*0.35 + hallucination*0.25

Results saved to amazon_data/
